# 🏋️‍♂️ YOLO Hand Shadow Dataset Training Notebook
สมุดโน้ตสำหรับใช้เทรนโมเดล YOLOv8/YOLO11 บน Google Colab ด้วยการใช้ GPU ฟรีในการเร่งความเร็วในการเทรน

---

## ⚙️ ขั้นตอนที่ 1: ตรวจสอบและตั้งค่าให้รันบน GPU
ก่อนเริ่มรัน กรุณาเข้าไปที่แถบเมนูด้านบน:
1. เลือก **Runtime** -> **Change runtime type**
2. ในช่อง **Hardware accelerator** เลือกเป็น **T4 GPU** (หรือ GPU ตัวอื่นที่มีให้เลือก)
3. กด **Save**

In [ ]:
# ตรวจสอบว่า Colab เปิดใช้งาน GPU สำเร็จหรือไม่
!nvidia-smi

## 📦 ขั้นตอนที่ 2: ติดตั้งไลบรารีที่จำเป็น
ติดตั้งแพ็คเกจ `ultralytics` สำหรับ YOLO และ `pyyaml` สำหรับจัดการไฟล์คอนฟิก

In [ ]:
!pip install ultralytics pyyaml

## 📤 ขั้นตอนที่ 3: อัปโหลดและแตกไฟล์ชุดข้อมูล `data.zip`
1. กดที่ไอคอน 📂 (Files) ด้านซ้ายมือ
2. ลากไฟล์ **`data.zip`** ที่ได้จากการรัน `prepare_dataset.py` ในเครื่องของคุณมาปล่อยเพื่ออัปโหลด
3. รอจนอัปโหลดไฟล์เสร็จสิ้น (วงกลมด้านล่างหยุดหมุน)
4. กดรันโค้ดเซลล์ด้านล่างนี้เพื่อแตกไฟล์ไปยังโฟลเดอร์ปลายทาง

In [ ]:
# แตกไฟล์ข้อมูลไปวางไว้ที่โฟลเดอร์หลัก
!unzip -q data.zip -d /content/

# ตรวจสอบโครงสร้างว่าโฟลเดอร์แตกเรียบร้อยดีหรือไม่
!ls -R /content/yolo_dataset

## 🏋️‍♂️ ขั้นตอนที่ 4: เริ่มเทรนโมเดล YOLO
เราจะเรียกใช้งานโมเดล YOLO เวอร์ชันล่าสุด โดยรันคำสั่งเทรนดึงค่าคอนฟิกจากไฟล์ `config.yaml` ที่เรา Unzip ขึ้นมา

> **💡 คำแนะนำ:** 
> * คุณสามารถเปลี่ยน `epochs=100` หรือเพิ่มขึ้นตามต้องการ (หากข้อมูลมีปริมาณน้อย 30-50 รอบอาจจะเร็วเกินไป แนะนำใช้ 100 รอบขึ้นไป)
> * `model="yolo11s.pt"` หรือ `model="yolov8s.pt"` เป็นโมเดลขนาดเล็กที่แม่นยำสูงและรันได้เร็วบนกล้องธรรมดา

In [ ]:
from ultralytics import YOLO

# โหลดโมเดล YOLO เริ่มต้น (โมเดลขนาดเล็ก 's' - Small)
model = YOLO("yolo11s.pt")

# เริ่มต้นเทรนดึงข้อมูลตามที่ระบุใน config.yaml
results = model.train(
    data="/content/yolo_dataset/config.yaml",
    epochs=100,
    imgsz=640,
    device=0,  # บอกให้รันบน GPU
    workers=2
)

## 📁 ขั้นตอนที่ 5: บีบอัดผลลัพธ์และดาวน์โหลดโมเดลกลับมาใช้งาน
หลังเทรนเสร็จสิ้นเสร็จ น้ำหนักโมเดลที่ดีที่สุดจะอยู่ที่ `runs/detect/train/weights/best.pt`
รันคำสั่งด้านล่างเพื่อบีบอัดและดาวน์โหลดตัวแปรโมเดลกลับมาใช้งานในโปรเจกต์คอมพิวเตอร์ของคุณ

In [ ]:
import os
from google.colab import files

# ค้นหาตำแหน่งไฟล์ best.pt ล่าสุดที่เพิ่งเทรนเสร็จ
best_model_path = "runs/detect/train/weights/best.pt"

if os.path.exists(best_model_path):
    print("กำลังดาวน์โหลดโมเดล best.pt ไปยังเครื่องคอมพิวเตอร์ของคุณ...")
    files.download(best_model_path)
else:
    # กรณีที่มีการเทรนหลายรอบ โฟลเดอร์อาจเปลี่ยนชื่อเป็น train2, train3 ...
    print("Error: ไม่พบไฟล์ในโฟลเดอร์เริ่มต้น กรุณาตรวจดูในโฟลเดอร์ runs/detect/")